In [ ]:
import pandas as pd

In [ ]:
!pip install fasttext

In [ ]:
!pip install numpy==1.26.4

In [ ]:
import os
import glob
import json
import re
import fasttext

In [ ]:
# ====== LOAD MODEL DETECT LANGUAGE ======
if not os.path.exists("lid.176.bin"):
    !wget -q https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin

model = fasttext.load_model("lid.176.bin")

def is_vietnamese(text):
    pred = model.predict(text.replace("\n", ""))
    return pred[0][0] == '__label__vi'

In [ ]:
# ====== REGEX CLEAN ======
def temp_clean_for_lang_detect(text):
    text = text.strip()

    # xử lý "said:"
    if "said:" in text:
        text = text.split("said:", 1)[1].strip()

    # tạm replace để tránh noise khi detect
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\b0\d{9,10}\b', ' ', text)
    text = re.sub(r'@\w+', ' ', text)

    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def final_clean(text):
    text = text.strip()

    if "said:" in text:
        text = text.split("said:", 1)[1].strip()

    return text


# ====== COUNT WORD ======
def count_words(text):
    # tách từ chuẩn hơn split()
    words = re.findall(r'\w+', text, flags=re.UNICODE)
    return len(words)

In [ ]:
import zipfile
zip_path = "/content/comment_txt-20260417T084922Z-3-001.zip"  # upload file zip lên Colab
extract_dir = "/content/data_extracted"
output_path = "/content/clean_data_voz.json"

# ====== UNZIP ======
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

# ====== TÌM FILE TXT ======
files = sorted(glob.glob(f"{extract_dir}/**/*.txt", recursive=True))

print(f"Tìm thấy {len(files)} file .txt")

Tìm thấy 30 file .txt


In [ ]:
removed_count = {
    "empty": 0,
    "not_vi": 0,
    "length": 0,
    "duplicate": 0
}

removed_samples = {k: [] for k in removed_count}
MAX_SAVE = 50

# ====== PROCESS ======
global_id = 1
seen_texts = set()

with open(output_path, 'w', encoding='utf-8') as out_f:
    for file in files:
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                raw_text = line.rstrip("\n")

                # EMPTY
                if not raw_text.strip():
                    removed_count["empty"] += 1
                    if len(removed_samples["empty"]) < MAX_SAVE:
                        removed_samples["empty"].append(raw_text)
                    continue

                # TEMP CLEAN
                temp_text = temp_clean_for_lang_detect(raw_text)

                if not is_vietnamese(temp_text):
                    removed_count["not_vi"] += 1
                    if len(removed_samples["not_vi"]) < MAX_SAVE:
                        removed_samples["not_vi"].append(raw_text)
                    continue

                # FINAL CLEAN
                text = final_clean(raw_text)

                # LENGTH
                word_count = count_words(text)
                if not (3 <= word_count <= 200):
                    removed_count["length"] += 1
                    if len(removed_samples["length"]) < MAX_SAVE:
                        removed_samples["length"].append(text)
                    continue

                # DEDUP
                norm_text = text.lower().strip()
                if norm_text in seen_texts:
                    removed_count["duplicate"] += 1
                    if len(removed_samples["duplicate"]) < MAX_SAVE:
                        removed_samples["duplicate"].append(text)
                    continue

                seen_texts.add(norm_text)

                # SAVE
                record = {
                    "id": f"voz_2022_{global_id}",
                    "text": text,
                    "length": word_count,
                    "source": "voz.vn"
                }

                out_f.write(json.dumps(record, ensure_ascii=False) + '\n')
                global_id += 1


# ====== REPORT ======
print(f"\nDone! Tổng sample giữ lại: {global_id - 1}")
print(f"File output: {output_path}")


Done! Tổng sample giữ lại: 11189
File output: /content/clean_data_voz.json


In [ ]:
print("\n===== THỐNG KÊ BỊ LOẠI =====")
for k, v in removed_count.items():
    print(f"{k}: {v}")


===== THỐNG KÊ BỊ LOẠI =====
empty: 0
not_vi: 109
length: 673
duplicate: 29


In [ ]:
print("\n===== SAMPLE NOT_VI =====")
for i, s in enumerate(removed_samples["not_vi"][:10], 1):
    print(f"\n--- Sample {i} ---")
    print(s)


===== SAMPLE NOT_VI =====

--- Sample 1 ---
https://voz.vn/t/nhat-ky-chay-bo-moi-ngay.167270/unread    via theNEXTvoz for iPhone

--- Sample 2 ---
Team mèo. 5 sao  via theNEXTvoz for iPhone

--- Sample 3 ---
Pics or didn't happen

--- Sample 4 ---
@kanywest 1 vote cho anh

--- Sample 5 ---
Lý do rớt giá:  Trong khi OPEC cắt giảm thì Nga lại tranh thủ tăng sản lượng: According to a Reuters survey, actual OPEC production in November amounted to 29.01 million bpd, some 710,000 bpd less than in October. The heavy lifting, according to the survey, was done by Saudi Arabia, which cut its November output by 500,000 bpd compared to October. Going completely against expectations, Russia, which, according to its own official figures and to data from energy intelligence firm Kpler, actually ramped up production in November. Ngoài ra G7 cũng sẽ chính thức tham gia vào áp trần giá dầu Nga, vậy ngày 4/12 tới đây rất có thể giá dầu sẽ biến động mạnh.

--- Sample 6 ---
via theNEXTvoz for iPhone

--- 

In [ ]:
print("\n===== SAMPLE DUPLICATE =====")
for s in removed_samples["duplicate"][:10]:
    print("-", s)


===== SAMPLE DUPLICATE =====
- soi cc, cho có ăn không
- Ranh giới đỏ đâu?  Gửi bằng vozFApp
- Chỗ Bà Triệu cũng có món lạc rang húng lìu y thế này, nhà nào cũng chính gốc ông bà gì đó
- @MasterchiefsReborn cái cờ jav bị gì vậy
- Spoiler: cân nhắc trước khi xem  tôi cảnh báo từ # trước rồi nhé
- Mới tí tuổi đã tập làm người xấu
- tính ra cần vẫn lành chán nhỉ, chơi đc ăn đc ngủ đc. 3 cái thứ ke, đá này hại vl
- Đeo kính như thầy thì đã ko bị đâm.
- Thằng này với thằng ml Đỗ quân chắc phải có trên tỷ đô, thằng nào cũng kêu lỗ thì ai lãi ở đây
- Nhiều con bò đi xe máy cứ thích cặp hông xe tải, thậm chí khi xe vào cua hay quay đầu hở có 0.5 mét cũng chen vào Đéo hiểu não để làm gì


In [ ]:
import json

input_path = "/content/clean_data_voz_unformated.json"
output_path = "/content/clean_data_voz.json"

In [ ]:
data = []

with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Done! Đã convert sang JSON list")

Done! Đã convert sang JSON list


In [ ]:
input_path = "/content/clean_data_voz.json"
output_path = "/content/clean_data_voz_dedup_prefix3.json"

In [ ]:
import json
import re

# ===== CLEAN FOOTER =====
def remove_footer(text):
    patterns = [
        r"\s*Sent using vozFApp\s*$",
        r"\s*Sent from .* using vozFApp\s*$",
        r"\s*Gửi bằng vozFApp\s*$",
        r"\s*via theNEXTvoz for .*?\s*$",
        r"\s*Sent from my .*?\s*$",
        r'View attachment \d+',
        r"\s*Gửi từ .*?\s*$",
        r"^\s*Gửi bằng vozFApp\s*",
    ]

    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    return text.strip()


# ===== REMOVE PREFIX =====
def remove_common_prefix(text1, text2, min_overlap=4):
    text1 = re.sub(r'\s+', ' ', text1.strip())
    text2 = re.sub(r'\s+', ' ', text2.strip())

    words1 = text1.split()
    words2 = text2.split()

    i = 0
    min_len = min(len(words1), len(words2))

    while i < min_len and words1[i] == words2[i]:
        i += 1

    if i >= min_overlap:
        return " ".join(words2[i:]).strip()

    return text2


# ===== PREFIX KEY =====
def get_prefix_key(text, n_words=12):
    text = re.sub(r'\s+', ' ', text.strip().lower())
    return " ".join(text.split()[:n_words])


# ===== LOAD =====
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)


# ===== PROCESS =====
window_size = 20
recent_texts = []
prefix_map = {}

new_data = []
prefix_removed_count = 0
footer_removed_count = 0

for item in data:
    raw_text = item["text"]

    # ===== STEP 1: REMOVE FOOTER =====
    text = remove_footer(raw_text)
    if text != raw_text:
        footer_removed_count += 1

    best_text = text

    # ===== STEP 2: GLOBAL PREFIX (xa) =====
    key = get_prefix_key(text)
    if key in prefix_map:
        prev_text = prefix_map[key]
        new_text = remove_common_prefix(prev_text, text)
        if new_text != text:
            best_text = new_text
            prefix_removed_count += 1

    # ===== STEP 3: LOCAL WINDOW (gần) =====
    else:
        for prev_text in reversed(recent_texts):
            new_text = remove_common_prefix(prev_text, text)
            if new_text != text:
                best_text = new_text
                prefix_removed_count += 1
                break

    # ===== SKIP EMPTY =====
    if not best_text.strip():
        continue

    # ===== UPDATE =====
    item["text"] = best_text
    item["length"] = len(best_text.split())

    new_data.append(item)

    # ===== UPDATE MEMORY =====
    recent_texts.append(best_text)
    if len(recent_texts) > window_size:
        recent_texts.pop(0)

    prefix_map[key] = best_text


# ===== SAVE =====
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)


# ===== REPORT =====
print("Done!")
print("Số sample:", len(new_data))
print("Prefix removed:", prefix_removed_count)
print("Footer removed:", footer_removed_count)
print("File:", output_path)

Done!
Số sample: 10752
Prefix removed: 3917
Footer removed: 2112
File: /content/clean_data_voz_dedup_prefix3.json


In [ ]:
import json
import re
import fasttext

# =========================
# LOAD FASTTEXT
# =========================
model = fasttext.load_model("lid.176.bin")

lang_cache = {}

def is_vietnamese(text):
    text = text.replace("\n", " ")
    if text in lang_cache:
        return lang_cache[text]

    pred = model.predict(text)
    result = pred[0][0] == "__label__vi"
    lang_cache[text] = result
    return result


# =========================
# COUNT WORDS (NO CLEAN)
# =========================
def count_words(text):
    return len(text.split())  # CHỈ SPLIT, KHÔNG CLEAN


# =========================
# INPUT / OUTPUT
# =========================
input_path = "/content/clean_data_voz_dedup_prefix3.json"
output_path = "/content/clean_data_voz_rechecked.json"


# =========================
# LOAD JSON
# =========================
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)


# =========================
# PROCESS
# =========================
new_data = []
stats = {
    "length_lt_3": 0,
    "not_vi": 0
}

lang_cache = {}

for item in data:
    text = item["text"]

    # 1. UPDATE LENGTH
    wc = count_words(text)
    item["length"] = wc

    # 2. FILTER LENGTH
    if wc < 3:
        stats["length_lt_3"] += 1
        continue

    # 3. FASTTEXT CHECK
    if not is_vietnamese(text):
        stats["not_vi"] += 1
        continue

    # 4. KEEP
    new_data.append(item)


# =========================
# SAVE
# =========================
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)


# =========================
# REPORT
# =========================
print("DONE")
print("Kept:", len(new_data))
print("Saved:", output_path)
print("Stats:", stats)

DONE
Kept: 10634
Saved: /content/clean_data_voz_rechecked.json
Stats: {'length_lt_3': 68, 'not_vi': 50}
